<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/11%20-%20Modelagem%20da%20Tubula%C3%A7%C3%A3o%20e%20Instrumentos%20como%20Grafo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2. Implementação da Classe Orientada a Objetos
A classe `GrafoNavegacaoAGV` permite gerenciar o mapa de rotas, distâncias e tags de verificação do AGV no galpão.

In [1]:
import pandas as pd
import numpy as np

class GrafoNavegacaoAGV:
    """
    Representação da malha logística do AGV autônomo via Dígrafo Ponderado G = (V, E, W).
    Mapeia waypoints no galpão, distâncias físicas e tags de verificação.
    """
    def __init__(self):
        self.estacoes = {}  # {tag_estacao: descricao}
        self.rotas = []     # Lista de arcos (origem, destino, distancia, tag_rfid)
        self._indices = {}  # Mapeamento interno tag -> índice da matriz

    def adicionar_estacao(self, tag_estacao: str, descricao: str):
        """Cadastra uma estação ou ponto de controle (Vértice v ∈ V)."""
        if tag_estacao not in self.estacoes:
            self._indices[tag_estacao] = len(self.estacoes)
            self.estacoes[tag_estacao] = descricao

    def adicionar_rota(self, origem: str, destino: str, distancia_m: float, tag_rfid: str):
        """Adiciona um corredor de navegação (Aresta e ∈ E) com peso W(e) em metros."""
        if origem not in self.estacoes or destino not in self.estacoes:
            raise KeyError("Estações de origem e destino devem ser cadastradas antes da rota.")

        self.rotas.append({
            'origem': origem,
            'destino': destino,
            'distancia_m': distancia_m,
            'tag_rfid': tag_rfid
        })

    def obter_matriz_adjacencia(self, pandas_format: bool = True):
        """
        Gera a Matriz de Adjacência Ponderada M, onde M[i, j] é a
        distância da estação i para a estação j (0 se não houver conexão direta).
        """
        n = len(self.estacoes)
        matriz = np.zeros((n, n))

        for rota in self.rotas:
            i = self._indices[rota['origem']]
            j = self._indices[rota['destino']]
            matriz[i, j] = rota['distancia_m']

        if pandas_format:
            tags = list(self.estacoes.keys())
            return pd.DataFrame(matriz, index=tags, columns=tags)
        return matriz

    def listar_rotas(self):
        """Retorna a tabela de corredores e rotas cadastradas."""
        return pd.DataFrame(self.rotas)

## 3. Povoamento do Grafo do AGV Logístico
Aqui instanciamos o grafo, adicionamos as estações físicas e configuramos as rotas (distâncias e tags).

In [2]:
# Instanciação do Grafo
agv_mapa = GrafoNavegacaoAGV()

# 1. Cadastro dos Vértices (Waypoints do AGV)
pontos_galpao = [
    ("ST-01", "Estação Base e Carregamento"),
    ("DOC-101", "Doca de Recebimento de Insumos"),
    ("ALM-201", "Almoxarifado Central"),
    ("AMO-301", "Posto de Amostragem de Reagentes"),
    ("R-101", "Reator Químico de Processo"),
    ("DEP-401", "Depósito Final e Expedição")
]

for tag, desc in pontos_galpao:
    agv_mapa.adicionar_estacao(tag, desc)

# 2. Cadastro dos Arcos (Rotas de Navegação + Distância + Tag LiDAR/RFID)
corredores = [
    ("ST-01", "DOC-101", 10.0, "TAG-01"),
    ("ST-01", "ALM-201", 12.0, "TAG-02"),
    ("DOC-101", "ALM-201", 15.0, "TAG-03"),
    ("ALM-201", "AMO-301", 20.0, "TAG-04"),
    ("AMO-301", "R-101", 18.0, "TAG-05"),
    ("AMO-301", "DEP-401", 14.0, "TAG-06"),
    ("R-101", "DEP-401", 25.0, "TAG-07"),
    ("DEP-401", "ST-01", 30.0, "TAG-08")
]

for orig, dest, dist, tag in corredores:
    agv_mapa.adicionar_rota(orig, dest, dist, tag)

print("Grafo do AGV populado com sucesso!")

Grafo do AGV populado com sucesso!


## 4. Geração e Exibição da Matriz de Adjacência
Abaixo geramos a matriz de adjacência ponderada $M_{6 \times 6}$ que o algoritmo de navegação lerá para calcular trajetos.

In [3]:
# Exibição da Matriz de Navegação do AGV
df_matriz_agv = agv_mapa.obter_matriz_adjacencia()

# Mostra o DataFrame bonitinho no output do Colab
df_matriz_agv

,ST-01,DOC-101,ALM-201,AMO-301,R-101,DEP-401
ST-01,0.0,10.0,12.0,0.0,0.0,0.0
DOC-101,0.0,0.0,15.0,0.0,0.0,0.0
ALM-201,0.0,0.0,0.0,20.0,0.0,0.0
AMO-301,0.0,0.0,0.0,0.0,18.0,14.0
R-101,0.0,0.0,0.0,0.0,0.0,25.0
DEP-401,30.0,0.0,0.0,0.0,0.0,0.0
